# 1. Context

This notebook assesses reduction in WER in samples where WER >= 10%. This helps in identifying Model & Prompt Combination which could help in Post OCR Correction

# 2. Assumption

* TextBlock where correction is required is known apriori
   
   * This can be assessed via confidence value return by Surya OCR


# 3. Imports

In [1]:
import pandas as pd
import jiwer
from pathlib import Path
from collections import defaultdict
import sys

In [2]:
notebook_path = Path()
sys.path.append(str(notebook_path.resolve().parent.parent))

In [3]:
from src.evaluation.metrics import cer, wer
from src.common.constants import language_to_writing_system, writing_system_to_language
from src.common.utils import get_script_results

# 4. Loading Consolidated Results

## 4.1. Loading Results

In [4]:
# get csv paths for all language
results_root = Path("../../results/surya_ocr")
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [ ]:
writing_sys_dict = defaultdict(list)
for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_dict[script].append(language_res)

script_language_result = pd.Series(writing_sys_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

In [6]:
script_results_avail = script_language_result.index
results_consolidated = []
for script in script_results_avail:
    results_script = get_script_results(script=script, script_lang_df=script_language_result, result_root=results_root)
    results_consolidated.append(results_script)
consolidated_df = pd.concat(results_consolidated)

## 4.2. Adding CER and WER

In [8]:
# computing cer & wer
gt_col = 'ground_truth'
ocr_output_cols = ['ocr_output_L_0', 'ocr_output_L_1', 'ocr_output_L_2', 'ocr_output_L_3']
cols_create_cer = ['cer_l0', 'cer_l1', 'cer_l2', 'cer_l3']
cols_create_wer = ['wer_l0', 'wer_l1', 'wer_l2', 'wer_l3']
for ocr_output_lvl, col_crt_cer in dict(zip(ocr_output_cols, cols_create_cer)).items():
    consolidated_df[col_crt_cer] = consolidated_df[[gt_col, ocr_output_lvl]].apply(lambda x: cer(x[gt_col], x[ocr_output_lvl]), axis=1)
for ocr_output_lvl, col_crt_wer in dict(zip(ocr_output_cols, cols_create_wer)).items():
    consolidated_df[col_crt_wer] = consolidated_df[[gt_col, ocr_output_lvl]].apply(lambda x: wer(x[gt_col], x[ocr_output_lvl]), axis=1)

In [10]:
col_ord = ['file_id', 'language', 'script','ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3', 'cer_l0', 'cer_l1', 'cer_l2',
       'cer_l3', 'wer_l0', 'wer_l1', 'wer_l2', 'wer_l3' ]
consolidated_df = consolidated_df[col_ord]

## upper casing column names
consolidated_df.columns = consolidated_df.columns.str.upper()

# 5. Selecting Text Blocks For Analysis

In [19]:
_LANGUGAE_EXCLUDE_ = ["santali", "manipuri"]
results_lang_scoped = consolidated_df[~consolidated_df['LANGUAGE'].isin(_LANGUGAE_EXCLUDE_)]

In [32]:
# degradation level & ocr output col map
degradation_ocr_col_map = {
    'L0': 'ocr_output_L_0',
    'L1': 'ocr_output_L_1',
    'L2': 'ocr_output_L_2',
    'L3': 'ocr_output_L_3'
}

In [46]:
filtered_rows_list = []
common_cols = ['FILE_ID', 'LANGUAGE', 'GROUND_TRUTH']
WER_COLS = ['WER_L0', 'WER_L1', 'WER_L2', 'WER_L3']
CER_COLS = ['CER_L0', 'CER_L1', 'CER_L2', 'CER_L3']
for idx, row in results_lang_scoped.iterrows():
    for CER in CER_COLS:
        if row[CER] >= 0.1:
            degradation_id = CER.split('_')[-1]
            ocr_output_col = degradation_ocr_col_map[degradation_id].upper()
            selected_values = row[common_cols + [ocr_output_col] + [CER]]
            selected_values.rename({ocr_output_col: 'OCR_OUTPUT', CER: 'CER'}, inplace=True)
            selected_values['HIGH_CER'] = True
            filtered_rows_list.append(selected_values)
    # for WER in WER_COLS:
    #     if row[WER] >= 0.1:
    #         degradation_id = WER.split('_')[-1]
    #         ocr_output_col = degradation_ocr_col_map[degradation_id].upper()
    #         selected_values = row[common_cols + [ocr_output_col] + [WER]]
    #         selected_values.rename({ocr_output_col: 'OCR_OUTPUT', WER: 'WER'}, inplace=True)
    #         selected_values['HIGH_WER'] = True
    #         filtered_rows_list.append(selected_values)
    

In [50]:
df_sample = pd.DataFrame(filtered_rows_list)

In [56]:
print(df_sample.loc[df_sample['LANGUAGE'] == "hindi"].iloc[0].GROUND_TRUTH)

यह में स्नातक यहाँ में इन्द्रप्रस्थ महिला महाविद्यालय जिसे इंद्रप्रस्थ कॉलेज या आईपी कॉलेज भी कहा जाता है, की स्‍थापना १९२४ में हुई थी। यह दिल्‍ली विश्‍वविद्यालय का सबसे पुराना महिला महाविद्यालय है। इसका आरंभ चांदनी चौक के जामा मसजिद क्षेत्र के छिप्‍पीवाड़ा में एक पुरानी हवेली की दूसरी मंजिल स्थित कमरे में तीन छात्राओं से हुआ। १९३० में स्नातक पाठ्यक्रम आरंभ हुए एवं १९३८ में विश्‍वविद्यालय द्वारा इन्‍द्रप्रस्‍थ महाविद्यालय को स्नातक महाविद्यालय के रूप में मान्‍यता मिली। कुछ वर्ष पश्‍चात् यह महाविद्यालय सिविल लाइन्‍स क्षेत्र के चन्‍द्रावली भवन में स्‍थानांतरित कर दिया गया और तदुपरांत १९३८ में इसे ब्रिेटिश कमांडर-इन-चीफ के अलीपुर रोड (वर्तमान शाम नाथ मार्ग) स्थित अलीपुर हाउस वाले कार्यालय-सह-आवास में पुन: स्‍थानांतरित कर दिया गया। इतिहास. थियोसॉफिकल सोसाइटी ऑफ इंडिया से संबद्ध समाजसेवियों के प्रयासों से मूल विद्यालय और महाविद्यालय विकसित हुआ। उन्‍हें थियोसॉफिस्‍ट श्रीमती एनी बेसेंट से प्रेरणा मिली थी। एनी बेसेंट ने उत्तर भारत की महिलाओं को शिक्षित करने का उस समय बीड़ा उठाया ज‍बकि महिलाएँ 

In [57]:
print(df_sample.loc[df_sample['LANGUAGE'] == "hindi"].iloc[0].OCR_OUTPUT)

यह में स्नातक यहाँ में में हुई थी। यह दिल्ली विश्वविद्यालय जिसे इंद्रप्रस्थ कॉलेज या आईपी कॉलेज भी कहा जाता है, की स्थापना १९२४ मसेजिद क्षेत्र के छिप्पीवाड़ा में एक पुरानी हवेली की दूसरी मंजिल स्थित कमरे में तीन छात्राओं से हुआ। १९२० में मसेजिद क्षेत्र के छिप्पीवाड़ा में एक पुरानी हवेली की दूसरी मंजिल स्थित कमरे में तीन छात्राओं से हुआ। १९३० में स्नातक पाठ्यक्रम आरंभ हुए एवं १९३८ में विश्वविद्यालय द्वारा इन्द्रप्रस्थ महाविद्यालय को सबाये पे पुराना महिला महाविद्यालय के अर्थ महावाद्यालय के जामा रूप में मान्यता मिली। कुछ वर्ष पश्चात् यह महाविद्यालय सिविल लाइन्स क्षेत्र के चन्द्रावली मतक महाविद्यालय के दिया गया और तदुपरांत १९३८ में इसे ब्रिटिश कमांडर-इन-चीफ के अलीपुर रोड (वर्तमान शाम नाथ मार्ग) स्थित अलीपुर हाउस वाले कार्यालय-सह-आवास में पुन: स्थानांतरित कर दिया गया। थियोसॉफिकल सोसाइटी ऑफ इंडिया से संबद्ध समाजसेवियों के प्रयासों से मूल विद्यालय और महाविद्यालय विकसित हुआ। उन्हें थियोसॉफिस्ट श्रीमती एनी बेसेंट् से प्रुरणा मिली थी। एनी बेसेंट ने उत्तर भारत की महिलाओं को शिक्षित करने का उस समय